# 04 — Feature Preparation

**Changes vs original:**

1. **Size-quintile neutralisation.** The dataset has no sector codes, so we use market-cap size
   quintiles as a neutralisation proxy. Each cross-section (date) is split into 5 quintiles by
   `avg_dollar_vol_1m` (a stable size proxy available every month). Fundamental features are
   z-scored *within* quintile × date cells rather than across the whole cross-section. This
   prevents the model from learning size tilts disguised as fundamental alpha.

2. **Leak-free target clipping.** `target_1m` is clipped using quantiles computed on the
   **training window only** (dates before `TRAIN_END`), then the same bounds are applied to
   validation and test rows. This stops future return distributions from influencing the
   training data.

3. **Leak-free feature standardisation.** Imputation (median fill) and z-scoring use
   cross-sectional statistics that are computed date-by-date from the data itself, so no
   time-series future leaks. The size-quintile assignment also uses only same-date peers.

In [1]:
import pandas as pd
import numpy as np
import json



# The training cutoff used to compute target-clipping bounds.
# Must match the earliest test fold start used in 05_model / 06_XGB_Regressor.
TRAIN_END = "2020-01-01"

## 1. Load

In [2]:
monthly = pd.read_csv(
    r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\monthly_features.csv",
    parse_dates=["Date"]
)

with open(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processed\feature_cols.json") as f:
    feature_cols = json.load(f)

target_col = "target_1m"

monthly = monthly.dropna(subset=[target_col])

print("Shape after dropping missing target:", monthly.shape)

Shape after dropping missing target: (114554, 90)


## 2. Leak-free target clipping

Compute clip bounds from training rows only; apply to all rows.

In [3]:
train_mask   = monthly["Date"] < TRAIN_END
target_lower = monthly.loc[train_mask, target_col].quantile(0.01)
target_upper = monthly.loc[train_mask, target_col].quantile(0.99)

monthly[target_col] = monthly[target_col].clip(target_lower, target_upper)

print(f"Target clip bounds (from train): [{target_lower:.4f}, {target_upper:.4f}]")
print(monthly[target_col].describe())

Target clip bounds (from train): [-0.2534, 0.3511]
count    114554.000000
mean          0.006927
std           0.098540
min          -0.253426
25%          -0.047349
50%           0.003556
75%           0.054650
max           0.351085
Name: target_1m, dtype: float64


## 3. Cross-sectional median imputation

Same as original — fill NaNs with the cross-sectional median for each date.

In [4]:
monthly[feature_cols] = (
    monthly
    .groupby("Date")[feature_cols]
    .transform(lambda x: x.fillna(x.median()))
)
monthly[feature_cols] = monthly[feature_cols].fillna(0)

print("Remaining NaNs after imputation:", monthly[feature_cols].isna().sum().sum())

Remaining NaNs after imputation: 0


## 4. Size-quintile neutralisation

No sector codes are available in this dataset. We use size (measured by `avg_dollar_vol_1m`)
as the neutralisation dimension. For each date, stocks are split into 5 quintiles by
dollar-volume. Fundamental and profitability features are then z-scored within each
quintile so that cross-quintile level differences do not drive model predictions.

Price/momentum/volatility features are z-scored across the full cross-section as before,
since their cross-size variation is itself a signal (small-cap effect, liquidity premium).

**Features neutralised within size quintile:**
all profitability, growth, and forecast features.

**Features z-scored cross-sectionally (full universe):**
momentum, MA ratios, volatility, skewness, dollar volume, relative volume.

In [5]:
SIZE_PROXY = "avg_dollar_vol_1m"
N_QUINTILES = 5

# Assign size quintile within each date
monthly["size_quintile"] = (
    monthly
    .groupby("Date")[SIZE_PROXY]
    .transform(
        lambda x: pd.qcut(x.rank(method="first"), q=N_QUINTILES, labels=False)
    )
)

# Split features into two buckets
PRICE_EXACT = {
    "log_market_cap",
    "price_to_earnings",
    "earnings_yield",
    "dividend_yield",
    "forecast_dividend_yield",
    "treasury_share_ratio",
}

PRICE_PREFIXES = [
    "mom_",
    "ma_",
    "vol_",
    "skew_",
    "max_return_",
    "avg_dollar_vol_",
    "rel_volume",
    "AdjustmentFactor", "SupervisionFlag",
]

PRICE_FEATURES = [
    f for f in feature_cols
    if (
        f in PRICE_EXACT or
        any(f.startswith(prefix) for prefix in PRICE_PREFIXES)
    )
]

FUNDAMENTAL_FEATURES = [f for f in feature_cols if f not in PRICE_FEATURES]

print(f"Price/technical features  ({len(PRICE_FEATURES)}): z-scored cross-sectionally")
print(f"Fundamental features      ({len(FUNDAMENTAL_FEATURES)}): z-scored within size quintile")

Price/technical features  (21): z-scored cross-sectionally
Fundamental features      (58): z-scored within size quintile


In [6]:
# Z-score fundamental features within [Date x size_quintile]
monthly[FUNDAMENTAL_FEATURES] = (
    monthly
    .groupby(["Date", "size_quintile"])[FUNDAMENTAL_FEATURES]
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
)

# Z-score price/technical features within [Date] (full cross-section)
monthly[PRICE_FEATURES] = (
    monthly
    .groupby("Date")[PRICE_FEATURES]
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
)

print("Final shape:", monthly.shape)

Final shape: (114554, 91)


## 5. Drop helper column and save

In [7]:
monthly = monthly.drop(columns=["size_quintile"])

monthly.to_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\model_data.csv", index=False)
print("Saved: model_data.csv —", monthly.shape)

Saved: model_data.csv — (114554, 90)
